# MedVis Exercise Sheet 2 — Task 3: Image Preprocessing / Smoothing

## Preparation

In [ ]:
!pip install scipy
!pip install pydicom

import os
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
from pydicom import dcmread

## Salt & Pepper Noise Function (provided — do not modify)

In [ ]:
def add_salt_pepper_noise(img):
    img_copy = img.copy()
    row, col = img_copy.shape
    salt_vs_pepper = 0.5
    amount = 0.04
    num_salt = np.ceil(amount * img_copy.size * salt_vs_pepper)
    num_pepper = np.ceil(amount * img_copy.size * (1.0 - salt_vs_pepper))
    # Add Salt noise
    coords = [np.random.randint(0, i - 1, int(num_salt)) for i in img_copy.shape]
    img_copy[tuple(coords)] = 900
    # Add Pepper noise
    coords = [np.random.randint(0, i - 1, int(num_pepper)) for i in img_copy.shape]
    img_copy[tuple(coords)] = 0
    return img_copy

## Read Image and Apply Noise

In [ ]:
# Read one DICOM slice and apply salt & pepper noise
dcm_slice = dcmread('dataset1/brain_011.dcm')
original_img = np.array(dcm_slice.pixel_array)
noisy_img = add_salt_pepper_noise(original_img)

## Apply Median and Gaussian Filters

In [ ]:
# Median filter — size=3 means a 3x3 neighborhood
median_filtered = ndimage.median_filter(noisy_img, size=3)

# Gaussian filter — sigma controls the amount of smoothing
gaussian_filtered = ndimage.gaussian_filter(noisy_img, sigma=1)

## Plot Results

In [ ]:
fig = plt.figure(figsize=(15, 5))

ax1 = fig.add_subplot(1, 4, 1)
ax1.set_title('Original')
ax1.imshow(original_img, cmap='gray')
ax1.axis('off')

ax2 = fig.add_subplot(1, 4, 2)
ax2.set_title('Noisy Image (Salt & Pepper)')
ax2.imshow(noisy_img, cmap='gray')
ax2.axis('off')

ax3 = fig.add_subplot(1, 4, 3)
ax3.set_title('Median Filter (size=3)')
ax3.imshow(median_filtered, cmap='gray')
ax3.axis('off')

ax4 = fig.add_subplot(1, 4, 4)
ax4.set_title('Gaussian Filter (sigma=1)')
ax4.imshow(gaussian_filtered, cmap='gray')
ax4.axis('off')

plt.tight_layout()
plt.show()

## Task 3 — Discussion

### Which filter works better for salt & pepper noise?
The **Median filter** works significantly better for salt & pepper noise. Salt & pepper noise produces extreme outlier values (very bright or very dark pixels). The median filter replaces each pixel with the **middle value** of its neighborhood, so these outliers are discarded entirely. The Gaussian filter uses a **weighted average**, which means the extreme noise values still influence the result, just spread out — producing a blurry image that still shows artifacts.

### When is the Median filter suited / not suited?
- **Suited:** Salt & pepper noise, preserving sharp edges, binary or segmented images
- **Not suited:** Images with thin structures (e.g. blood vessels) — the median can remove thin lines entirely if they are narrower than the kernel size

### When is the Gaussian filter suited / not suited?
- **Suited:** Gaussian (random) noise spread across the image, pre-processing before edge detection, general smoothing
- **Not suited:** Salt & pepper noise (extreme outliers still influence the average), images where edge sharpness must be preserved

### Typical medical imaging artifacts reduced by these filters
- **Gaussian:** Reduces general scanner noise (thermal noise in MRI)
- **Median:** Reduces spike artifacts, random bright/dark pixel errors from detector malfunctions